In [1]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('plot_paper')

enss=['b','c','d','e']

# gluon

In [2]:
folder='step1'
key2phy_A20_pre=yu.load_pkl_reg(f'{folder}/key2phy_A20_stout',pathlabel='analysis_xJ_syst')
key2phy_B20_pre=yu.load_pkl_reg(f'{folder}/key2phy_B20_stout',pathlabel='analysis_xJ_syst')
stouts_jointlinear=[7,10,13,15,20]

In [19]:
stouts=stouts_jointlinear

key2phy_A20={(ens,f'jg;stout{nst}'):key2phy_A20_pre[(ens,f'jg;stout{nst}')] for ens in enss for nst in stouts }
for nst in stouts:
    key2phy_A20[('a=#_const',f'jg;stout{nst}')]=key2phy_A20_pre[('a=#_const',f'jg;stout{nst}')]
    key2phy_A20[('a=#_linear',f'jg;stout{nst}')]=key2phy_A20_pre[('a=#_linear',f'jg;stout{nst}')]
    key2phy_A20[('a=#_MA',f'jg;stout{nst}')]=key2phy_A20_pre[('a=#_MA',f'jg;stout{nst}')]

y_jk=yu.superjackknife([np.transpose([key2phy_A20[(ens,f'jg;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

fig,axs=yu.getFigAxs(1,2,sharey=True)
ax=axs[0,0]
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    if nst==10:
        phy_A20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_A20[(ens,f'jg;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=nst if iens==0 else None)
        
    t=key2phy_A20[('a=#_linear',f'jg;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt)
    
# for nst in [10]:
#     mean,err=yu.jackme(key2phy_A20[('a=#_linear',f'jg;stout{nst}')])
#     x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
#     ax.fill_between(x, ymin, ymax, color='black', alpha=0.1)

ax.set_xlim([-0.001,0.0075])
ax.legend(ncols=2)

ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007], xticklabels=[0,1,2,3,4,5,6,7])
ax.set_ylabel(r'$\langle x \rangle_g$')

ax=axs[0,1]
ens2phy={}
for iens,ens in enumerate(enss):
    ens2phy[ens]=np.transpose([key2phy_A20_pre[(ens,f'jg;stout{nst}')] for nst in stouts])
    # print(ens,[yu.jackme_un2str(key2phy_A20_pre[(ens,f'jg;stout{nst}')]) for nst in stouts])
    pars_jk,chi2_jk,Ndof=yu.doFit_const(ens2phy[ens],corrQ=False)
    # print(yu.jackme_un2str(pars_jk[:,0]))
    ens2phy[ens]=pars_jk[:,0]
    mean,err=yu.jackme(pars_jk)
    plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr, color='black')
fits=yu.doFits_continuumExtrapolation(ens2phy,lat_a2s_plt=yum.lat_a2s_plt,fitlabels=['linear'])
pars_jk,probs_jk=yu.jackMA(fits)
print(yu.jackme_un2str(pars_jk[:,0]))
mean,err=yu.jackme(pars_jk)
x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
ax.fill_between(x, ymin, ymax, color='black', alpha=0.1)
ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
yu.addRefLine(ax,0,'v')
yu.finalizePlot('jointLinear_avgx')

0.392(51) 1.0347765983618562 14
0.375(57)


In [18]:
stouts=stouts_jointlinear

key2phy_B20={(ens,f'jg;stout{nst}'):key2phy_B20_pre[(ens,f'jg;stout{nst}')] for ens in enss for nst in stouts }
for nst in stouts:
    key2phy_B20[('a=#_const',f'jg;stout{nst}')]=key2phy_B20_pre[('a=#_const',f'jg;stout{nst}')]
    key2phy_B20[('a=#_linear',f'jg;stout{nst}')]=key2phy_B20_pre[('a=#_linear',f'jg;stout{nst}')]
    key2phy_B20[('a=#_MA',f'jg;stout{nst}')]=key2phy_B20_pre[('a=#_MA',f'jg;stout{nst}')]

y_jk=yu.superjackknife([np.transpose([key2phy_B20[(ens,f'jg;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

fig,axs=yu.getFigAxs(1,2,sharey=True)
ax=axs[0,0]
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    if nst==10:
        phy_B20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_B20[(ens,f'jg;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,label=nst if iens==0 else None)
        
    t=key2phy_B20[('a=#_linear',f'jg;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt)

ax.set_xlim([-0.001,0.0075])
ax.legend(ncols=2)

ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007], xticklabels=[0,1,2,3,4,5,6,7])
ax.set_ylabel(r'$B_{20}^{g}$')

yu.addRefLine(ax,0)

ax=axs[0,1]
ens2phy={}
for iens,ens in enumerate(enss):
    ens2phy[ens]=np.transpose([key2phy_B20_pre[(ens,f'jg;stout{nst}')] for nst in stouts])
    pars_jk,chi2_jk,Ndof=yu.doFit_const(ens2phy[ens],corrQ=False)
    ens2phy[ens]=pars_jk[:,0]
    mean,err=yu.jackme(pars_jk)
    plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr, color='black')
fits=yu.doFits_continuumExtrapolation(ens2phy,lat_a2s_plt=yum.lat_a2s_plt,fitlabels=['linear'])
pars_jk,probs_jk=yu.jackMA(fits)
print(yu.jackme_un2str(pars_jk[:,0]))
mean,err=yu.jackme(pars_jk)
x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
ax.fill_between(x, ymin, ymax, color='black', alpha=0.1)

ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(-0.0007, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
yu.addRefLine(ax,0,'v')
yu.finalizePlot(f'jointLinear_B20')

0.036(51) 1.943756140195582 14
0.075(58)


# others

In [5]:
folders=['step1','case3','step2','case2']

key2phy_A20s=[yu.load_pkl_reg(f'{folder}/key2phy_A20',pathlabel='analysis_xJ_syst') for folder in folders]
key2phy_B20s=[yu.load_pkl_reg(f'{folder}/key2phy_B20',pathlabel='analysis_xJ_syst') for folder in folders]
key2phy_Js=[yu.load_pkl_reg(f'{folder}/key2phy_J',pathlabel='analysis_xJ_syst') for folder in folders]

In [6]:
ens2Njk={'b':725,'c':400,'d':493,'e':516}
enss=['b','c','d']
ens2q2gA={
    'b':{'ju':[0.867,0.018],'jd':[-0.408,0.014],'js':[-0.0262,0.0086],'jc':[0.0003,0.0045],'jq':[0.433,0.030],'j-':[1.275,0.023],'j+;conn':[0.585,0.013]},
    'c':{'ju':[0.832,0.022],'jd':[-0.415,0.021],'js':[-0.036,0.017],'jc':[-0.012,0.012],'jq':[0.368,0.065],'j-':[1.247,0.015],'j+;conn':[0.567,0.011]},
    'd':{'ju':[0.843,0.016],'jd':[-0.415,0.015],'js':[-0.034,0.013],'jc':[0.0076,0.0094],'jq':[0.401,0.049],'j-':[1.258,0.014],'j+;conn':[0.5659,0.0072]}
}
ens2rb={ens:{} for ens in enss}
for ens in enss:
    for j in ['jq','ju','jd','js','jc','j+;conn','j-']:
        ens2rb[ens][f'{j}']=yu.jackknife_pseudo([ens2q2gA[ens][j][0]],np.array([[ens2q2gA[ens][j][1]**2+1e-10]]),ens2Njk[ens])[:,0]
    ens2rb[ens][f'ju;conn']=(ens2rb[ens][f'j+;conn']+ens2rb[ens][f'j-'])/2
    ens2rb[ens][f'jd;conn']=(ens2rb[ens][f'j+;conn']-ens2rb[ens][f'j-'])/2
    ens2rb[ens][f'jq;conn']=ens2rb[ens][f'j+;conn']
    
ens2rb['a=#_final']={}

a2s_plt=yum.lat_a2s_plt
for j in ['jq','ju','jd','js','jc','ju;conn','jd;conn','jq;conn']:
    a2s=np.array([yu.ens2a[ens]**2 for ens in enss])
    y_jk=yu.superjackknife([ens2rb[ens][f'{j}'][:,None] for ens in enss])
        
    fits=[]
    
    def fitfunc(pars):
        g=pars
        return g+0*a2s
    pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,[1])
    def fitfunc_plt(pars):
        g=pars
        return g+0*a2s_plt
    pars_plt=yu.jackmap(fitfunc_plt,pars_jk)
    # ens2rb['a=#_const'][f'{j}']=pars_plt
    fits.append(['const',pars_plt,chi2_jk,Ndof])
    
    def fitfunc(pars):
        g,c=pars
        return g+c*a2s
    pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,[1,1])
    def fitfunc_plt(pars):
        g,c=pars
        return g+c*a2s_plt
    pars_plt=yu.jackmap(fitfunc_plt,pars_jk)
    # ens2rb['a=#_linear'][f'{j}']=pars_plt
    fits.append(['linear',pars_plt,chi2_jk,Ndof])
    
    pars_plt,props_jk=yu.jackMA(fits,systematicQ=True)
    ens2rb['a=#_final'][f'{j}']=pars_plt

# for ft in ['const','linear','MA']:
#     ens2rb[f'a=#_{ft}'][f'jtot;{stout}']=ens2rb[f'a=#_{ft}'][f'jq;{stout}']+ens2rb[f'a=#_{ft}'][f'jg;{stout}']
#     ens2rb[f'a=#_{ft}'][f'jtot;conn;{stout}']=ens2rb[f'a=#_{ft}'][f'jq;conn;{stout}']

#     for fla in fla2iso.keys():
#         ens2rb[f'a=#_{ft}'][f'j{fla};{stout}']=np.sum([factor*ens2rb[f'a=#_{ft}'][f'j{iso};{stout}'] for factor,iso in fla2iso[fla]],axis=0)
#     for fla in fla2iso_conn.keys():
#         ens2rb[f'a=#_{ft}'][f'j{fla};conn;{stout}']=np.sum([factor*ens2rb[f'a=#_{ft}'][f'j{iso};conn;{stout}'] for factor,iso in fla2iso_conn[fla]],axis=0)

key2phy={}
for ens in ens2rb.keys():
    for j in ens2rb[ens].keys():
        t=ens2rb[ens][j]
        if ens not in enss:
            m,e=yu.jackme(t)
            t=yu.jackknife_pseudo(m,e,sum([ens2Njk[ens] for ens in ['b','c','d','e']]))
        key2phy[(ens,j)]=t
key2phy_gA=key2phy
enss=['b','c','d','e']

key2phy_DeltaSigmaBy2={key:key2phy_gA[key]/2 for key in key2phy_gA.keys()}
key2phy_L={key:key2phy_Js[0][key]-key2phy_DeltaSigmaBy2[key] for key in key2phy_DeltaSigmaBy2.keys() if key[1] not in ['j+;conn','j-']}

for j,j1 in zip(['jtot','jtot;conn'],['jq','jq;conn']):
    ens='a=#_final'
    key2phy_DeltaSigmaBy2[(ens,j)]=key2phy_DeltaSigmaBy2[(ens,j1)]
    key2phy_L[(ens,j)]=key2phy_L[(ens,j1)]

In [20]:
def get_key2syst(key2phys):
    key2syst={}
    for key in key2phys[0].keys():
        if 'conn' in key[1]:
            continue
        m0=np.mean(key2phys[0][key],axis=0)
        ds=np.array([np.mean(key2phy[key],axis=0)-m0 for key2phy in key2phys])
        t=np.sum(ds[1:]**2,axis=0)
        key2syst[key]=np.sqrt(t)
    return key2syst

def get_key2syst_2(key2phys):
    key2syst={}
    for key in key2phys[0].keys():
        if 'conn' in key[1]:
            continue
        m0=np.mean(key2phys[0][key],axis=0)
        ds=np.array([np.mean(key2phy[key],axis=0)-m0 for key2phy in key2phys])
        t1=np.sum(ds[1:3]**2,axis=0)
        t2=np.sum(ds[3:]**2,axis=0)
        key2syst[key]=(np.sqrt(t1),np.sqrt(t2))
    return key2syst


def plot_A20_B20_J(fig,axs,key2phy,which,ylabelQ=True,ce='final',key2syst=None,syst0onlyQ=True,rightmostQ=False):
    sty = {"jtot": ("gray", "o"), "jq": ("purple", "d"), "jg": ("cyan", "s"),
           "ju": ("red", "^"), "jd": ("green", "v"), "js": ("blue", "<"), "jc": ("orange", ">")}

    if which == "A20":
        rows = [(["jtot", "jq", "jg"], r"$\langle x\rangle_{q,g}$", (0.20, 1.40), [0.4,0.6,0.8,1.0,1.2], 1.0),
                (["ju", "jd", "js", "jc"], r"$\langle x\rangle_q$", (-0.10, 0.58), [0.0, 0.2, 0.4], 0.0)]
        lab = dict(jtot=r"$\langle x\rangle_N$", jq=r"$\langle x\rangle_q$", jg=r"$\langle x\rangle_g$",
                   ju=r"$\langle x\rangle_u$", jd=r"$\langle x\rangle_d$",
                   js=r"$\langle x\rangle_s$", jc=r"$\langle x\rangle_c$")
    elif which == "B20":
        rows = [(["jtot"], r"$B_{20}^N$", (-0.5, 0.5), [-0.3, 0.0, 0.3], 0.0),
                (["jq", "jg"], r"$B_{20}^{q,g}$", (-0.3, 0.3), [-0.2, 0.0, 0.2], 0.0),
                (["ju", "jd", "js", "jc"], r"$B_{20}^q$", (-0.22, 0.22), [-0.1, 0.0, 0.1], 0.0)]
        lab = {j: rf"$B_{{20}}^{{{s}}}$" for j, s in
               zip(["jtot", "jq", "jg", "ju", "jd", "js", "jc"], ["N", "q", "g", "u", "d", "s", "c"])}
    else:
        rows = [(["jtot", "jq", "jg"], r"$J_{q,g}$", (0.03, 0.80), [0.1, 0.3, 0.5, 0.7], 0.5),
                (["ju", "jd", "js", "jc"], r"$J_q$", (-0.03, 0.35), [0.0, 0.1, 0.2, 0.3], 0.0)]
        lab = {j: rf"$J_{{{s}}}$" for j, s in
               zip(["jtot", "jq", "jg", "ju", "jd", "js", "jc"], ["N", "q", "g", "u", "d", "s", "c"])}

    for ax, (js, ylabel, ylim, yticks, ref) in zip(axs, rows):
        for ij,j in enumerate(js):
            c, m = sty[j]
            x = np.asarray(yum.lat_a2s_plt)
            y, e = map(np.asarray, yu.jackme(key2phy[("a=#_" + ce, j)]))

            ax.plot(x, y, "--", color=c, lw=2)
            ax.fill_between(x, y - e, y + e, color=c, alpha=0.2)

            xs, ys, es = zip(*[(yu.ens2a[ens]**2 + 0.001/10*ij, *yu.jackme(key2phy[(ens, j)])) for iens,ens in enumerate(enss)])
            ax.errorbar(xs, ys, yerr=es, fmt=m, color=c, label=lab[j], capsize=6, lw=2, ms=6)
            
            if key2syst is not None:
                x = np.asarray(yum.lat_a2s_plt)
                y, e = map(np.asarray, yu.jackme(key2phy[("a=#_" + ce, j)]))
                if syst0onlyQ:
                    ax.errorbar(0.001/5,y[0],e[0], color=c, fmt='*', markersize=16, mfc='white')
                    e = np.sqrt(e**2 + key2syst[("a=#_" + ce, j)]**2)
                    ax.errorbar(0.001/5,y[0],e[0], color=c, fmt='*', markersize=16, mfc='white')
                else:
                    e = np.sqrt(e**2 + key2syst[("a=#_" + ce, j)]**2)
                    ax.plot(x, y, "--", color=c, lw=2)
                    ax.fill_between(x, y - e, y + e, color=c, alpha=0.1)

                    xs, ys, es = zip(*[(yu.ens2a[ens]**2+ 0.001/10*ij, *yu.jackme(key2phy[(ens, j)])) for iens,ens in enumerate(enss)])
                    es=np.sqrt(np.array(es)**2 + np.array([key2syst[(ens, j)] for ens in enss])**2)
                    ax.errorbar(xs, ys, yerr=es, fmt=m, color=c, capsize=6, lw=2, ms=6)

        ax.axhline(ref, color="black", ls=":", lw=2, marker="")
        ax.set(ylabel=ylabel if ylabelQ else None, ylim=ylim)
        ax.set(ylabel=ylabel if ylabelQ else None, ylim=ylim, yticks=yticks)
        # ax.tick_params(direction="in", top=True, right=True)
        # ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=4))

        if which=='B20' and j=='jtot':
            pass
        else:
            ax.legend(loc="upper right", frameon=True, ncol=len(js),
                  fontsize=15, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

    ax=axs[-1]
    if rightmostQ:
        ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(0, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
    else:
        ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(0, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007], xticklabels=[0,1,2,3,4,5,6,7])
    
def plot_A20_B20_J_v1(fig,axs,key2phy,which,ylabelQ=True,ce='final',key2syst=None,syst0onlyQ=True,rightmostQ=False):
    sty = {"jv1": ("r", "s")}

    if which == "A20":
        rows = [(['jv1'], r"$\langle x\rangle_{v1}$", (0.12,0.22), [0.14,0.16,0.18,0.2], 1.0)]
        lab = dict(jv1=r"$\langle x\rangle_{v1}$")
    elif which == "B20":
        rows = [(['jv1'], r"$B_{20}^{v1}$", (0.1,0.3), [0.15,0.2,0.25], 1.0)]
        lab = dict(jv1=r"$B_{20}^{v1}$")
    else:
        rows = [(['jv1'], r"$J_{v1}$", (0.1,0.3), [0.15,0.2,0.25], 1.0)]
        lab = dict(jv1=r"$J_{v1}$")

    for ax, (js, ylabel, ylim, yticks, ref) in zip(axs, rows):
        for ij,j in enumerate(js):
            c, m = sty[j]
            x = np.asarray(yum.lat_a2s_plt)
            y, e = map(np.asarray, yu.jackme(key2phy[("a=#_" + ce, j)]))

            ax.plot(x, y, "--", color=c, lw=2)
            ax.fill_between(x, y - e, y + e, color=c, alpha=0.2)

            xs, ys, es = zip(*[(yu.ens2a[ens]**2 + 0.001/10*ij, *yu.jackme(key2phy[(ens, j)])) for iens,ens in enumerate(enss)])
            ax.errorbar(xs, ys, yerr=es, fmt=m, color=c, label=lab[j], capsize=6, lw=2, ms=6)
            
            if key2syst is not None:
                x = np.asarray(yum.lat_a2s_plt)
                y, e = map(np.asarray, yu.jackme(key2phy[("a=#_" + ce, j)]))
                if syst0onlyQ:
                    ax.errorbar(0.001/5,y[0],e[0], color=c, fmt='*', markersize=16, mfc='white')
                    e = np.sqrt(e**2 + key2syst[("a=#_" + ce, j)]**2)
                    ax.errorbar(0.001/5,y[0],e[0], color=c, fmt='*', markersize=16, mfc='white')
                else:
                    e = np.sqrt(e**2 + key2syst[("a=#_" + ce, j)]**2)
                    ax.plot(x, y, "--", color=c, lw=2)
                    ax.fill_between(x, y - e, y + e, color=c, alpha=0.1)

                    xs, ys, es = zip(*[(yu.ens2a[ens]**2+ 0.001/10*ij, *yu.jackme(key2phy[(ens, j)])) for iens,ens in enumerate(enss)])
                    es=np.sqrt(np.array(es)**2 + np.array([key2syst[(ens, j)] for ens in enss])**2)
                    ax.errorbar(xs, ys, yerr=es, fmt=m, color=c, capsize=6, lw=2, ms=6)

        ax.axhline(ref, color="black", ls=":", lw=2, marker="")
        ax.set(ylabel=ylabel if ylabelQ else None, ylim=ylim, yticks=yticks)
        ax.tick_params(direction="in", top=True, right=True)
        
    ax=axs[-1]
    if rightmostQ:
        ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(0, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007])
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3), useMathText=True)
    else:
        ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(0, 0.0075), xticks=[0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007], xticklabels=[0,1,2,3,4,5,6,7])
        
def run(key2phys,which,ratios,figsize):
    fig, axs = plt.subplots(1, len(folders), figsize=(6*len(folders),4), sharex='col', sharey='row',
                            gridspec_kw={"hspace": 0, 'wspace': 0.02})
    for i in range(len(folders)):
        plot_A20_B20_J_v1(fig,[axs[i]],key2phys[i],which,ylabelQ=(i==0),rightmostQ=(i==3))
    yu.finalizePlot(f'a2dep_{which}_compare_v1',tightQ=False)
    
    fig, axs = plt.subplots(len(ratios), len(folders), figsize=(figsize[0]*len(folders),figsize[1]), sharex='col', sharey='row',
                            gridspec_kw={"height_ratios": ratios, "hspace": 0, 'wspace': 0.02})
    for i in range(len(folders)):
        plot_A20_B20_J(fig,axs[:,i],key2phys[i],which,ylabelQ=(i==0),rightmostQ=(i==3))
    yu.finalizePlot(f'a2dep_{which}_compare',tightQ=False)
    
    key2phy=key2phys[0]; key2syst=get_key2syst(key2phys); key2syst_2=get_key2syst_2(key2phys)
    
    fig, axs = plt.subplots(len(ratios), 1, figsize=figsize, sharex=True, sharey='row',
                            gridspec_kw={"height_ratios": ratios, "hspace": 0, 'wspace': 0.02})
    plot_A20_B20_J(fig,axs,key2phy,which,rightmostQ=True)
    yu.finalizePlot(f'a2dep_{which}',tightQ=False)
    
    fig, axs = plt.subplots(len(ratios), 1, figsize=figsize, sharex=True, sharey='row',
                            gridspec_kw={"height_ratios": ratios, "hspace": 0, 'wspace': 0.02})
    plot_A20_B20_J(fig,axs,key2phy,which,key2syst=key2syst,rightmostQ=True)
    yu.finalizePlot(f'a2dep_{which}_syst',tightQ=False)
    
    js=['jtot','jq','jv1','jv2','jv3','jg','ju','jd','js','jc']
    index={
        'A20':[r'$\braket{x}'+(rf'_{{{j[1:]}}}$' if j not in ['jtot'] else r'_{N}$' ) for j in js],
        'B20':[r'$B_{20}'+(rf'^{{{j[1:]}}}$' if j not in ['jtot'] else r'^{N}$' ) for j in js],
        'J':[r'$J'+(rf'_{{{j[1:]}}}$' if j not in ['jtot'] else r'_{N}$' ) for j in js]
    }[which]
    
    columns=[yu.ens2label[ens] for ens in enss] + [r'$a=0$']
    df=[[yu.jackme_un2str(key2phy[(ens,j)],precision=2)for ens in enss] 
        + [yu.me2mes(yu.jackme_un2str(key2phy[('a=#_final',j)][:,0],precision=2), (key2syst_2[('a=#_final',j)][0][0],key2syst_2[('a=#_final',j)][1][0]))]  for j in js]
    df=pd.DataFrame(df,index=index,columns=columns)
    tex=df.to_latex(
        escape=False,          # allow latex in labels
        column_format='cccccc',
        bold_rows=False,
    )
    tex = tex.replace(r'\toprule'+'\n',r'')
    tex = tex.replace(r'\midrule', r'\hline')
    tex = tex.replace(r'\bottomrule'+'\n',r'')
    print(tex)

In [21]:
run(key2phy_A20s,'A20',[1.15, 1.0], (5.6, 7.4))
run(key2phy_B20s,'B20',[1.0, 1.0, 2.0], (5.6, 7.4))
run(key2phy_Js,'J',[1.15, 1.0], (5.6, 7.4))

\begin{tabular}{cccccc}
 & B64 & C80 & D96 & E112 & $a=0$ \\
\hline
$\braket{x}_{N}$ & 0.973(59) & 0.981(65) & 0.965(99) & 1.04(14) & 1.015(75)(24)(91) \\
$\braket{x}_{q}$ & 0.590(47) & 0.551(48) & 0.661(71) & 0.66(10) & 0.623(46)(22)(91) \\
$\braket{x}_{v1}$ & 0.158(15) & 0.170(11) & 0.164(11) & 0.1638(98) & 0.1649(71)(96)(7) \\
$\braket{x}_{v2}$ & 0.493(40) & 0.517(29) & 0.495(24) & 0.401(38) & 0.450(29)(11)(48) \\
$\braket{x}_{v3}$ & 0.577(41) & 0.599(30) & 0.571(26) & 0.477(41) & 0.523(32)(10)(44) \\
$\braket{x}_{g}$ & 0.382(25) & 0.430(36) & 0.303(54) & 0.373(69) & 0.392(51)(22)(0) \\
$\braket{x}_{u}$ & 0.357(24) & 0.359(20) & 0.377(23) & 0.355(33) & 0.357(17)(11)(34) \\
$\braket{x}_{d}$ & 0.199(21) & 0.189(17) & 0.214(21) & 0.191(30) & 0.192(16)(2)(35) \\
$\braket{x}_{s}$ & 0.0313(66) & 0.0156(98) & 0.048(17) & 0.072(24) & 0.049(12)(5)(10) \\
$\braket{x}_{c}$ & 0.0034(53) & -0.0121(81) & 0.023(15) & 0.047(21) & 0.025(10)(6)(12) \\
\end{tabular}

\begin{tabular}{cccccc}
 & B64 & C

In [ ]:
def plot_bar(key2phy,which,key2syst=None):
    colors = dict(ju="red", jd="green", js="blue", jc="orange",
                  jq="purple", jg="cyan", jtot="gray")
    names = dict(ju=r"$u$", jd=r"$d$", js=r"$s$", jc=r"$c$",
                 jq=r"$q$", jg=r"$g$", jtot=r"$\mathrm{Total}$")

    setup = {
        "A20": (["ju", "jd", "js", "jc", "jq", "jg", "jtot"],
                r"$\langle x\rangle_{q,g}$", (0.0, 1.25),
                [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2], [1.0], 1.0, 2),
        "J": (["ju", "jd", "js", "jc", "jq", "jg", "jtot"],
              r"$J_{q,g}$", (0.0, 0.62),
              [0.0, 0.1, 0.2, 0.3, 0.4, 0.5], [0.5], 0.5, 0),
        "DeltaSigmaBy2": (["ju", "jd", "js", "jc", "jtot"],
                          r"$\frac{1}{2}\Delta\Sigma_q$", (-0.32, 0.57),
                          [-0.25, 0.0, 0.25, 0.5], [0.0, 0.5], 0.5, -16),
        "L": (["ju", "jd", "js", "jc", "jtot"],
              r"$L_{q,g}$", (-0.32, 0.57),
              [-0.25, 0.0, 0.25, 0.5], [0.0, 0.5], 0.5, -16),
    }

    js, ylabel, ylim, yticks, refs, norm, labelpad = setup[which]
    yr = ylim[1] - ylim[0]
    x = np.arange(len(js))

    fig, ax = plt.subplots(figsize=(7.2, 5.4))

    for i, j in enumerate(js):
        c = colors[j]

        m, e = yu.jackme(key2phy[("a=#_final", j)][:, 0])
        ax.bar(i, m, width=0.52, color=c, alpha=0.22, edgecolor=c, linewidth=1.3)
        ax.errorbar(i, m, yerr=e, fmt="none", color="black", capsize=6, lw=2)
        if key2syst is not None:
            m, e = yu.jackme(key2phy[("a=#_final", j)][:, 0])
            e = np.sqrt(e**2 + np.array([key2syst[("a=#_final", j)][0]])**2)[0]
            ax.errorbar(i, m, yerr=e, fmt="none", color="black", capsize=6, lw=2)

        jc = f"{j};conn"
        if ("a=#_final", jc) in key2phy:
            mc, _ = yu.jackme(key2phy[("a=#_final", jc)][:, 0])
            ax.bar(i, mc, width=0.32, color=c, alpha=0.85, edgecolor=c, linewidth=1.3)

        txt = rf"${100*m/norm:.1f}({100*e/norm:.1f})\%$"
        if key2syst is not None:
            m, e = yu.jackme(key2phy[("a=#_final", j)][:, 0])
            s = key2syst[("a=#_final", j)][0]
            txt = rf"${100*m/norm:.1f}({100*e/norm:.1f})({100*s/norm:.1f})\%$"
        
        ytxt = m + np.sign(m if m else 1) * (e + 0.035 * yr)
        ytxt = np.clip(ytxt, ylim[0] + 0.18 * yr, ylim[1] - 0.18 * yr)

        ax.text(i - 0.42, ytxt, txt, rotation=90,
                ha="center", va="center", fontsize=12, clip_on=True)

    for y in refs:
        ax.axhline(y, color="black", ls="--", lw=2, marker="")

    ax.set(
        ylabel=ylabel, ylim=ylim, yticks=yticks,
        xticks=x, xticklabels=[names[j] for j in js],
    )
    ax.set_xlim(-0.75, len(js) - 0.25)
    ax.yaxis.labelpad = labelpad
    ax.tick_params(direction="in", top=True, right=True)

    for s in ax.spines.values():
        s.set_linewidth(2)

    t={'A20':'avgx'}[which] if which in ['A20'] else which
    yu.finalizePlot(f'bar_{t}')
    return fig, ax

for which in ["A20", "J", "DeltaSigmaBy2", "L"][:2]:
    plot_bar(globals()[f"key2phy_{which}s"][0],which,key2syst=get_key2syst(globals()[f"key2phy_{which}s"]))

# compare

In [ ]:
pdflabels=['HERAPDF2.0','ABMP16','CT18','MSHT20','NNPDF4.0','PDF4LHC21','JAM22','CJ22']
pdfsets=['HERAPDF20_NNLO_EIG','ABMP16_4_nnlo','CT18NNLO','MSHT20nnlo_as118','NNPDF40_nnlo_as_01180','PDF4LHC21_40_pdfas','JAM22-PDF_proton_nlo','CJ22']
label2j2me_A20=yu.load_pkl_reg('label2j2me_avgx',pathlabel='lhapdf')
label2j2me_A20={label:{j[:-2] if j.endswith(';+') else j:me for j,me in label2j2me_A20[label].items()} for label in label2j2me_A20.keys()}

pdflabels_pol=['DSSV14','JAM22','NNPDFpol2.0']
pdfsets_pol=['DSSV14pol','JAM22-PPDF_proton_nlo','NNPDFpol20_nnlo_as_01180_mhou']
label2j2me_gA=yu.load_pkl_reg('label2j2me_gA_xmin=1e-3',pathlabel='lhapdf')
label2j2me_gA={label:{j[:-2] if j.endswith(';+') else j:me for j,me in label2j2me_gA[label].items()} for label in label2j2me_gA.keys()}

label2j2me_DeltaSigmaBy2={label:{j:(m/2,e/2) for j,(m,e) in label2j2me_gA[label].items()} for label in label2j2me_gA.keys()}

In [ ]:
def plot_compare(which):
    key2phy = globals()[f"key2phy_{which}s"][0]
    get_src = getattr(yum, f"get_{which}_from_src")
    
    key2syst = get_key2syst(globals()[f"key2phy_{which}s"])

    cfg = {
        "A20": dict(
            js=["ju", "jd", "js", "jc", "jg"],
            xlabel=[r"$\langle x\rangle_u$", r"$\langle x\rangle_d$", r"$\langle x\rangle_s$",
                    r"$\langle x\rangle_c$", r"$\langle x\rangle_g$"],
            phen=(["ABMP16", "CT18", "MSHT20", "NNPDF4.0", "JAM22", "CJ22"],
                  pdflabels, pdfsets, label2j2me_A20),
            srcs=[r"$\chi$QCD18", "MIT24", "ETM20"],
            xlim=[(0.22, 0.52), (0.06, 0.36), (-0.10, 0.20), (-0.12, 0.18), (0.10, 0.60)],
            xticks=[[0.30, 0.45], [0.15, 0.30], [0.0, 0.1], [0.0, 0.1], [0.25, 0.45]],
            figsize=(13.5, 3.5),
        ),
        "J": dict(
            js=["ju", "jd", "js", "jc", "jg"],
            xlabel=[r"$J_u$", r"$J_d$", r"$J_s$", r"$J_c$", r"$J_g$"],
            phen=None, srcs=["MIT24", "ETM20"],
            xlim=[(0.12, 0.32), (-0.04, 0.16), (-0.08, 0.12), (-0.09, 0.09), (0.10, 0.30)],
            xticks=[[0.18, 0.28], [0.0, 0.1], [0.0, 0.08], [0.0, 0.07], [0.15, 0.25]],
            figsize=(13.5, 1.1),
        ),
        
        "DeltaSigmaBy2": dict(
            js=["ju", "jd", "js", "jc"],
            xlabel=[r"$\frac{1}{2}\Delta\Sigma_u$", r"$\frac{1}{2}\Delta\Sigma_d$",
                    r"$\frac{1}{2}\Delta\Sigma_s$", r"$\frac{1}{2}\Delta\Sigma_c$"],
            phen=(["DSSV14", "JAM22", "NNPDFpol2.0"],
                  pdflabels_pol, pdfsets_pol, label2j2me_DeltaSigmaBy2),
            srcs=[r"$\chi$QCD18", "PNDME25", "Mainz26", "ETM20"],
            xlim=[(0.32, 0.52), (-0.31, -0.11), (-0.12, 0.10), (-0.09, 0.09)],
            xticks=[[0.35, 0.45], [-0.25, -0.15], [-0.08, 0.0], [-0.06, 0.0, 0.06]],
            figsize=(11.5, 2.8),
        ),
        "L": dict(
            js=["ju", "jd", "js", "jc"],
            xlabel=[r"$L_u$", r"$L_d$", r"$L_s$", r"$L_c$"],
            phen=None, srcs=["ETM20"],
            xlim=[(-0.30, -0.10), (0.16, 0.36), (-0.04, 0.16), (-0.09, 0.09)],
            xticks=[[-0.25, -0.15], [0.2, 0.3], [0.0, 0.1], [0.0, 0.07]],
            figsize=(11.5, 0.8),
        ),
    }[which]

    def missing(v):
        return v is None or (isinstance(v, tuple) and any(x is None for x in v))

    def split_err(e):
        if isinstance(e, tuple):
            estat, esys = e
            return estat, np.sqrt(estat**2 + esys**2)
        return None, e

    phen_rows = []
    if cfg["phen"] is not None:
        order, pdflabs, pdfsets_, label2j2me = cfg["phen"]
        lab2set = dict(zip(pdflabs, pdfsets_))
        phen_rows = [(lab, lab2set[lab]) for lab in order if lab in lab2set]

    ylabels = [lab for lab, _ in phen_rows] + cfg["srcs"] + ["This Work"]
    y = np.arange(len(ylabels))
    ymap = dict(zip(ylabels, y))

    fig, axs = plt.subplots(1, len(cfg["js"]), figsize=cfg["figsize"], sharey=True, gridspec_kw={"wspace": 0.05})
    axs = np.atleast_1d(axs)

    for ax, j, xl, xlim, xticks in zip(axs, cfg["js"], cfg["xlabel"], cfg["xlim"], cfg["xticks"]):
        m, e = yu.jackme(key2phy[("a=#_final", j)][:, 0])
        
        ax.errorbar(m, ymap["This Work"], xerr=e, fmt="s", color="red")
        etot = np.sqrt(e**2 + key2syst[("a=#_final", j)][0]**2)
        ax.errorbar(m, ymap["This Work"], xerr=etot, fmt="s", color="red")
        
        ax.axvspan(m - etot, m + etot, color="red", alpha=0.20)
        # ax.axvline(m, color="red", alpha=0.45)
    
        if cfg["phen"] is not None:
            label2j2me = cfg["phen"][3]
            for lab, label in phen_rows:
                if which == "DeltaSigmaBy2" and j == "jc" and lab in ["DSSV14", "JAM22"]:
                    continue
                val = label2j2me.get(label, {}).get(j)
                if missing(val):
                    continue
                mp, ep = val
                if missing((mp, ep)):
                    continue
                _, ep = split_err(ep)
                ax.errorbar(mp, ymap[lab], xerr=ep, fmt="^", color="black")

        for src in cfg["srcs"]:
            val = get_src(src, j)
            if missing(val):
                continue
            ms, es = val
            if missing((ms, es)):
                continue
            estat, etot = split_err(es)

            if src == "ETM20":
                col, mk, mfc = "green", "o", "white"
            elif src == "MIT24":
                col, mk, mfc = "blue", "d", "white"
            else:
                col, mk, mfc = "blue", "d", "blue"

            ax.errorbar(ms, ymap[src], xerr=etot, fmt=mk, color=col, mfc=mfc, mec=col)
            if estat is not None:
                ax.errorbar(ms, ymap[src], xerr=estat, fmt="none", color=col)

        ax.set(xlabel=xl, xlim=xlim, xticks=xticks)
        ax.set_ylim(len(ylabels) - 0.25, -0.75)
        ax.tick_params(direction="in", top=True, right=True, labelsize=16)
        for s in ax.spines.values():
            s.set_linewidth(2)

    axs[0].set_yticks(y)
    axs[0].set_yticklabels(ylabels, fontsize=18)
    for ax in axs[1:]:
        ax.tick_params(labelleft=False)

    t={'A20':'avgx'}[which] if which in ['A20'] else which
    yu.finalizePlot(f'compare_{t}',tightQ=False)
    return fig, axs


for which in ["A20", "J", "DeltaSigmaBy2", "L"][:2]:
    plot_compare(which)